# MoodTune — Phase 4: Mood Labeling Methodology

This notebook creates reproducible pseudo-labels, not verified human emotional ground truth. It compares clustering diagnostics with a percentile-based, data-derived prototype method.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

project_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((root for root in project_roots if (root / 'data' / 'processed').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the MoodTune project or a child directory.')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.config.mood_config import MOOD_FEATURES, RANDOM_SEED, SONG_MOODS, assign_pseudo_moods, build_seed_masks

songs = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'spotify_cleaned.csv')
display(songs.loc[:, MOOD_FEATURES].quantile([.25, .33, .50, .67, .75]))

## Clustering diagnostic

K-Means is tested after standardization because the features use different units. The elbow and silhouette values diagnose structure, but clusters are not automatically treated as moods.

In [ ]:
scaled = StandardScaler().fit_transform(songs.loc[:, MOOD_FEATURES])
sample = songs.sample(n=min(10000, len(songs)), random_state=RANDOM_SEED)
sample_scaled = StandardScaler().fit_transform(sample.loc[:, MOOD_FEATURES])
cluster_results = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    labels = model.fit_predict(sample_scaled)
    cluster_results.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(sample_scaled, labels, sample_size=min(5000, len(sample)), random_state=RANDOM_SEED)})
cluster_results = pd.DataFrame(cluster_results)
display(cluster_results)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cluster_results['k'], cluster_results['inertia'], marker='o')
axes[0].set(title='K-Means Elbow Diagnostic', xlabel='k', ylabel='Inertia')
axes[1].plot(cluster_results['k'], cluster_results['silhouette'], marker='o')
axes[1].set(title='K-Means Silhouette Diagnostic', xlabel='k', ylabel='Silhouette score')
plt.tight_layout()
plt.show()

diagnostic_model = KMeans(n_clusters=5, random_state=RANDOM_SEED, n_init=10)
songs['diagnostic_cluster'] = diagnostic_model.fit_predict(scaled)
display(songs.groupby('diagnostic_cluster')[list(MOOD_FEATURES)].mean().round(3))
songs = songs.drop(columns='diagnostic_cluster')

## Hybrid pseudo-label generation

High-confidence, overlapping seed groups use dataset percentiles and interpretable feature combinations. Their observed mean profiles become prototypes. Every song receives the mood of its nearest standardized prototype, avoiding arbitrary fixed score cut-offs and preserving all tracks for later modeling.

In [ ]:
moods, seed_profiles, quantiles = assign_pseudo_moods(songs)
seed_masks = build_seed_masks(songs, quantiles)
seed_counts = pd.Series({mood: int(mask.sum()) for mood, mask in seed_masks.items()}, name='seed_count').reindex(list(SONG_MOODS))
labeled = songs.assign(mood=moods)
class_counts = labeled['mood'].value_counts().reindex(list(SONG_MOODS)).rename('track_count')
class_summary = labeled.groupby('mood')[['valence', 'energy', 'danceability', 'acousticness', 'tempo']].mean().reindex(list(SONG_MOODS))
display(pd.concat([seed_counts, class_counts, (class_counts / len(labeled) * 100).round(2).rename('percentage')], axis=1))
display(seed_profiles.round(3))
display(class_summary.round(3))

OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'spotify_mood_labeled.csv'
labeled.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote {len(labeled):,} labeled tracks to {OUTPUT_PATH}')

In [ ]:
from sklearn.metrics import pairwise_distances_argmin
feature_scaler = StandardScaler().fit(labeled.loc[:, MOOD_FEATURES])
representatives = []
for mood in SONG_MOODS:
    group = labeled[labeled['mood'].eq(mood)]
    center = group.loc[:, MOOD_FEATURES].mean().to_frame().T
    position = pairwise_distances_argmin(feature_scaler.transform(group.loc[:, MOOD_FEATURES]), feature_scaler.transform(center))[0]
    representatives.append(group.iloc[position][['mood', 'track_name', 'artists', 'track_genres']])
display(pd.DataFrame(representatives))

fig, axis = plt.subplots(figsize=(10, 4))
class_counts.plot.bar(ax=axis, color='#5B8FF9')
axis.set(title='Pseudo-Mood Class Distribution', xlabel='Mood', ylabel='Tracks')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()